In [ ]:
import os

INPUT_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8' # Adjust if the folder name is slightly different

def print_directory_tree(startpath, max_depth=2):
    print(f"Scanning: {startpath}")
    for root, dirs, files in os.walk(startpath):
        depth = root[len(startpath):].count(os.sep)
        if depth > max_depth: continue
        indent = ' ' * 4 * depth
        print(f"{indent}📁 {os.path.basename(root)}/ ({len(files)} files)")
        if files:
            for f in files[:3]: print(f"{indent}    📄 {f}")
            if len(files) > 3: print(f"{indent}    ... and {len(files) - 3} more.")

print_directory_tree(INPUT_DIR)

In [ ]:
import os

# Define our new paths based on your screenshot
BASE_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8/dataset'
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'Images/train')
TRAIN_LBL_DIR = os.path.join(BASE_DIR, 'labels/train')

# Get the first text file
sample_label_file = os.listdir(TRAIN_LBL_DIR)[0]
sample_label_path = os.path.join(TRAIN_LBL_DIR, sample_label_file)

print(f"Reading label file: {sample_label_file}\n")
with open(sample_label_path, 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        print(line.strip())
        if i >= 4: # Print only the first 5 lesions to avoid spamming the output
            print("... and more lesions")
            break
            
print(f"\nTotal lesions in this image: {len(lines)}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def plot_yolo_bounding_boxes(image_path, label_path):
    # 1. Load the image using OpenCV (BGR format) and convert to RGB
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error loading image: {image_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_height, img_width = img.shape[:2]

    # 2. Read the labels
    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"No label file found for {image_path}")
        return

    # 3. Draw each bounding box
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5: continue
            
        class_id = int(parts[0])
        x_center_norm, y_center_norm, w_norm, h_norm = map(float, parts[1:])

        # Convert YOLO normalized coordinates back to absolute pixel values
        x_center = x_center_norm * img_width
        y_center = y_center_norm * img_height
        box_width = w_norm * img_width
        box_height = h_norm * img_height

        # Calculate top-left and bottom-right coordinates for OpenCV
        x_min = int(x_center - (box_width / 2))
        y_min = int(y_center - (box_height / 2))
        x_max = int(x_center + (box_width / 2))
        y_max = int(y_center + (box_height / 2))

        # Draw the rectangle (Red color, 2px thickness)
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)
        
        # Put the class label text
        cv2.putText(img, f"Class {class_id}", (x_min, y_min - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # 4. Plot the image
    plt.figure(figsize=(12, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Ground Truth Bounding Boxes\n{os.path.basename(image_path)} ({len(lines)} lesions)")
    plt.show()

# Let's visualize the exact image corresponding to the label file we looked at in Step 4
sample_img_file = sample_label_file.replace('.txt', '.jpg')
sample_img_path = os.path.join(TRAIN_IMG_DIR, sample_img_file)

plot_yolo_bounding_boxes(sample_img_path, sample_label_path)

# Data Engineering Pipeline

<h3>Step 1: The Box Tightening Pipeline<h3/>

In [1]:
import os
import shutil

# Paths based on your dataset structure
ORIGINAL_IMG_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8/dataset/Images/train'
ORIGINAL_LBL_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8/dataset/labels/train'

# Our new clean working directories
WORKING_DIR = '/kaggle/working/skinwise_data'
NEW_IMG_DIR = os.path.join(WORKING_DIR, 'images/train')
NEW_LBL_DIR = os.path.join(WORKING_DIR, 'labels/train')

# Create directories
os.makedirs(NEW_IMG_DIR, exist_ok=True)
os.makedirs(NEW_LBL_DIR, exist_ok=True)

SHRINK_FACTOR = 0.85 # Shrink boxes by 15%

def tighten_yolo_boxes():
    label_files = [f for f in os.listdir(ORIGINAL_LBL_DIR) if f.endswith('.txt')]
    print(f"Processing {len(label_files)} label files...")
    
    processed_count = 0
    for label_file in label_files:
        orig_label_path = os.path.join(ORIGINAL_LBL_DIR, label_file)
        new_label_path = os.path.join(NEW_LBL_DIR, label_file)
        
        # We also need to copy the corresponding image to our new directory
        img_file = label_file.replace('.txt', '.jpg')
        orig_img_path = os.path.join(ORIGINAL_IMG_DIR, img_file)
        new_img_path = os.path.join(NEW_IMG_DIR, img_file)
        
        if not os.path.exists(orig_img_path):
            continue # Skip if image is missing
            
        # Copy image
        shutil.copy(orig_img_path, new_img_path)
        
        # Read, shrink, and write labels
        new_lines = []
        with open(orig_label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = parts[0]
                    cx, cy, w, h = map(float, parts[1:])
                    
                    # Apply shrinkage to width and height only
                    new_w = w * SHRINK_FACTOR
                    new_h = h * SHRINK_FACTOR
                    
                    new_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {new_w:.6f} {new_h:.6f}\n")
                    
        with open(new_label_path, 'w') as f:
            f.writelines(new_lines)
            
        processed_count += 1
        
    print(f"✅ Successfully engineered {processed_count} images and labels with 15% tighter boxes!")

tighten_yolo_boxes()


Processing 1165 label files...
✅ Successfully engineered 1165 images and labels with 15% tighter boxes!


<h3>Step 2: Creating the YAML Configuration</h3>

In [2]:
yaml_content = f"""
path: /kaggle/working/skinwise_data
train: images/train
val: images/train # We are using train for val temporarily just to test the pipeline

names:
  0: acne
"""

with open('/kaggle/working/skinwise_data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml created successfully!")


✅ data.yaml created successfully!


<h3>Step 3: Train the Baseline YOLOv8 Model!<h3/>

In [3]:
# Install ultralytics (YOLOv8)
!pip install ultralytics

from ultralytics import YOLO

# Load a pre-trained YOLOv8 'small' model
model = YOLO('yolov8s.pt') 

print("🚀 Starting YOLOv8 Training...")
# Train the model on our newly engineered data
results = model.train(
    data='/kaggle/working/skinwise_data.yaml', 
    epochs=20,          # Just 20 epochs for our baseline test
    imgsz=640,          # Resize images to 640x640
    batch=16,           # Batch size of 16 fits nicely on Kaggle GPUs
    project='/kaggle/working/SkinWISE_Models',
    name='baseline_yolov8s'
)
print("✅ Baseline Training Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.9 MB/s eta 0:00:0000:0100:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Starting YOLOv8 Training...
Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/skinwise_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=Fals

<h2>Phase 2: Model Export & Sync
</h2>

<h3>Step 1: Exporting to ONNX</h3>

In [4]:
from ultralytics import YOLO

# Load the best weights from our baseline training run
model = YOLO('/kaggle/working/SkinWISE_Models/baseline_yolov8s/weights/best.pt')

print("⚙️ Exporting model to ONNX format...")
# Export the model. imgsz must match what we will receive from the web frontend
success = model.export(
    format='onnx', 
    dynamic=False, # Fixed size is faster for CPU
    imgsz=640,
    opset=17       # Industry standard ONNX opset
)

print(f"✅ Export complete: {success}")

⚙️ Exporting model to ONNX format...
Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/kaggle/working/SkinWISE_Models/baseline_yolov8s/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 329ms
Prepared 2 packages in 2.73s
Installed 2 packages in 11ms
 + onnxruntime-gpu==1.25.0
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 3.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 17...
ONNX: sli

<h3>Step 2: Packaging for Local Download</h3>

In [5]:
import shutil

# Zip the training results and the ONNX model
shutil.make_archive(
    '/kaggle/working/baseline_results', 
    'zip', 
    '/kaggle/working/SkinWISE_Models/baseline_yolov8s'
)
print("📦 baseline_results.zip created in your /kaggle/working/ directory.")
print("⬇️ You can now download it from the right-hand panel in Kaggle.")


📦 baseline_results.zip created in your /kaggle/working/ directory.
⬇️ You can now download it from the right-hand panel in Kaggle.
